# Loan Approval Prediction ML Pipeline

## Simple Architecture
### User Input → DataFrame → Preprocessor → Polynomial Features → Model → Prediction/

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import PolynomialFeatures, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

In [2]:
data = pd.read_csv("loan.csv")

In [3]:
data.isnull().sum()

Loan_ID               0
Gender                5
Married               0
Dependents            8
Education             4
Self_Employed        23
ApplicantIncome       4
CoapplicantIncome     5
LoanAmount            2
Loan_Amount_Term     13
Credit_History       30
Property_Area         1
Loan_Status           0
dtype: int64

In [4]:
data.Gender.fillna(data.Gender.mode()[0], inplace=True)
data.Dependents.fillna(data.Dependents.mode()[0], inplace=True)
data.Education.fillna(data.Education.mode()[0], inplace=True)
data.Self_Employed.fillna(data.Self_Employed.mode()[0], inplace=True)
data.ApplicantIncome.fillna(data.ApplicantIncome.mean(), inplace=True)
data.CoapplicantIncome.fillna(data.CoapplicantIncome.median(), inplace=True)
data.LoanAmount.fillna(data.LoanAmount.mean(), inplace=True)
data.Loan_Amount_Term.fillna(data.Loan_Amount_Term.mode()[0], inplace=True)
data.Credit_History.fillna(data.Credit_History.median(), inplace=True)
data.Property_Area.fillna(data.Property_Area.mode()[0], inplace=True)

C:\Users\Yousuf Traders\AppData\Local\Temp\ipykernel_3860\436438653.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data.Gender.fillna(data.Gender.mode()[0], inplace=True)
C:\Users\Yousuf Traders\AppData\Local\Temp\ipykernel_3860\436438653.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behave

In [5]:
data.isnull().sum()

Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64

In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 381 entries, 0 to 380
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            381 non-null    object 
 1   Gender             381 non-null    object 
 2   Married            381 non-null    object 
 3   Dependents         381 non-null    object 
 4   Education          381 non-null    object 
 5   Self_Employed      381 non-null    object 
 6   ApplicantIncome    381 non-null    float64
 7   CoapplicantIncome  381 non-null    float64
 8   LoanAmount         381 non-null    float64
 9   Loan_Amount_Term   381 non-null    float64
 10  Credit_History     381 non-null    float64
 11  Property_Area      381 non-null    object 
 12  Loan_Status        381 non-null    object 
dtypes: float64(5), object(8)
memory usage: 38.8+ KB


In [7]:
data.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001003,Male,Yes,1,Graduate,No,4583.0,1508.0,128.0,360.0,1.0,Rural,N
1,LP001005,Male,Yes,0,Graduate,Yes,3000.0,0.0,66.0,360.0,1.0,Urban,Y
2,LP001006,Male,Yes,0,Not Graduate,No,2583.0,2358.0,120.0,360.0,1.0,Urban,Y
3,LP001008,Male,No,0,Graduate,No,6000.0,0.0,141.0,360.0,1.0,Urban,Y
4,LP001013,Male,Yes,0,Not Graduate,No,2333.0,1516.0,95.0,360.0,1.0,Urban,Y


In [8]:
data.duplicated().sum()

np.int64(0)

In [9]:
data.drop_duplicates(inplace=True)
data.reset_index(drop=True, inplace=True)

In [10]:
X = data.drop(["LoanAmount", "Loan_ID", "Loan_Status", "Loan_Amount_Term"], axis=1)
y = data["LoanAmount"]

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

## Preprocessing Pipelines

In [12]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="median")),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="most_frequent")),
    ('encoder', OneHotEncoder(drop="first",handle_unknown="ignore", sparse_output=False))
])

## Column Transformer

In [13]:
preprocessor = ColumnTransformer([
    ('num', num_pipeline, make_column_selector(dtype_include=np.number)),
    ('cat', cat_pipeline, make_column_selector(dtype_include=object))
])

## Final Model Pipeline

In [14]:
model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('model', LinearRegression())
])

In [15]:
model_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('poly', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [16]:
y_pred = model_pipeline.predict(X_test)

In [17]:
r2 = r2_score(y_pred,y_test)
mae = mean_absolute_error(y_pred,y_test)

print(r2, mae)

-0.22409513433346295 27.276252065872374


## Comparision of Linear & Polynomial Regression 

In [18]:
linear_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

In [19]:
linear_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [20]:
linear_pred = linear_pipeline.predict(X_test)

In [21]:
linear_r2 = r2_score(y_test, linear_pred)
linear_mae = mean_absolute_error(y_test, linear_pred)

In [22]:
poly_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('model', LinearRegression())
])

In [23]:
poly_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('poly', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [24]:
poly_pred = poly_pipeline.predict(X_test)

In [25]:
poly_r2 = r2_score(y_test, poly_pred)
poly_mae = mean_absolute_error(y_test, poly_pred)

In [26]:
print(f"""
-----------------------------------------Model Prediction Which one is Better than the other-------------------------------------------------------
Linear R2 Score {linear_r2}
Linear Absolute Error {linear_mae}

Polynomial R2 Score {poly_r2}
Polynomial Absolute Error {poly_mae}""")


-----------------------------------------Model Prediction Which one is Better than the other-------------------------------------------------------
Linear R2 Score -0.15866412721322565
Linear Absolute Error 19.10288018488948

Polynomial R2 Score -2.367303425893106
Polynomial Absolute Error 27.276252065872374
